# Hugging Face Ready-to-Use Sentiment Analysis


## Objective

We use Hugging Face pretrained pipelines.

We do not train.

We directly use a ready-made sentiment model.

We use China mirror.

We use PyTorch backend.

We test on `imdb_top_500.csv`.

* 1 = Positive
* 0 = Negative

---

## Step 1: Install Libraries

In [ ]:
!pip install transformers pandas torch -q

---

## Step 2: Set Hugging Face Mirror

In [ ]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

---

## Step 3: Import Libraries

In [ ]:
import pandas as pd

from transformers import pipeline

---

## Step 4: Load Dataset

In [ ]:
df = pd.read_csv("./datasets/imdb_top_500.csv")

In [ ]:
print("Dataset size:", len(df))

print("Columns:", df.columns.tolist())

print("\nFirst review preview:")
print(df["text"].iloc[0][:300])

print("\nFirst label:", df["label"].iloc[0])

print("First rating:", df["rating"].iloc[0])

---

## Step 5: Build Hugging Face Pipeline

We use a stable pretrained model.

In [ ]:
classifier = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    tokenizer="distilbert-base-uncased-finetuned-sst-2-english",
    framework="pt"
)

In [ ]:
print("Pipeline loaded successfully.")

print("Model name:", classifier.model.name_or_path)

---

## Step 6: Check Label Style

In [ ]:
print("Model labels:")
print(classifier.model.config.id2label)

---

## Step 7: Test One Positive Sentence

In [ ]:
result = classifier("This movie was fantastic and exciting")

In [ ]:
print("Prediction:")
print(result)

---

## Step 8: Test One Negative Sentence

In [ ]:
result = classifier("This movie was terrible and boring")

In [ ]:
print("Prediction:")
print(result)

---

## Step 9: Predict First 10 IMDB Reviews

In [ ]:
sample_reviews = df["text"].iloc[:10].tolist()

In [ ]:
predictions = classifier(
    sample_reviews,
    truncation=True
)

In [ ]:
print("Total predictions:", len(predictions))

print("\nFirst prediction:")
print(predictions[0])

---

## Step 10: Convert POSITIVE and NEGATIVE into 1 and 0

In [ ]:
def hf_to_label(pred):
    return 1 if pred["label"] == "POSITIVE" else 0

In [ ]:
converted_preds = [
    hf_to_label(pred)
    for pred in predictions
]

In [ ]:
print("Converted predictions:")
print(converted_preds)

---

## Step 11: Compare with Real Labels

In [ ]:
true_labels = df["label"].iloc[:10].tolist()

In [ ]:
for i in range(10):
    print(f"\nReview {i+1}")
    print("-" * 50)

    print(df["text"].iloc[i][:200])

    print("\nTrue Label:", true_labels[i])

    print("Predicted Label:", converted_preds[i])

    if true_labels[i] == converted_preds[i]:
        print("Result: Correct")
    else:
        print("Result: Incorrect")

---

## Step 12: Run on First 50 Reviews

In [ ]:
subset_size = 50

reviews = df["text"].iloc[:subset_size].tolist()

true_labels = df["label"].iloc[:subset_size].tolist()

In [ ]:
predictions = classifier(
    reviews,
    truncation=True,
    batch_size=8
)

In [ ]:
pred_labels = [
    hf_to_label(pred)
    for pred in predictions
]

In [ ]:
print("Finished predicting", len(pred_labels), "reviews.")

---

## Step 13: Compute Accuracy

In [ ]:
correct = sum(
    p == y
    for p, y in zip(pred_labels, true_labels)
)

accuracy = correct / len(true_labels)

In [ ]:
print("Accuracy on first", subset_size, "reviews:", accuracy)

---

## Step 14: View Confidence Scores

In [ ]:
for i in range(5):
    print(f"\nReview {i+1}")

    print("HF Label:", predictions[i]["label"])

    print("Confidence Score:", predictions[i]["score"])

---

## Step 15: Compare Real Reviews More Clearly

In [ ]:
for i in range(5):
    print(f"\nReview {i+1}")
    print("-" * 60)

    print(reviews[i][:300])

    print("\nTrue Label:", true_labels[i])

    print("Predicted Label:", pred_labels[i])

    if true_labels[i] == pred_labels[i]:
        print("Result: Correct")
    else:
        print("Result: Incorrect")

---

## Step 16: Predict Custom Reviews

In [ ]:
custom_reviews = [
    "This movie was amazing with brilliant acting",
    "I hated this film so much",
    "It was okay, not bad, not great"
]

In [ ]:
custom_preds = classifier(
    custom_reviews,
    truncation=True
)

In [ ]:
for review, pred in zip(custom_reviews, custom_preds):
    print("\nReview:")
    print(review)

    print("HF Label:", pred["label"])

    print("Numeric Label:", hf_to_label(pred))

    print("Score:", pred["score"])

---

## Step 17: Simple Batch Prediction Function

In [ ]:
def predict_sentiment(text_list):
    preds = classifier(
        text_list,
        truncation=True
    )

    return [
        {
            "text": text,
            "hf_label": pred["label"],
            "numeric_label": hf_to_label(pred),
            "score": pred["score"]
        }
        for text, pred in zip(text_list, preds)
    ]

In [ ]:
sample_results = predict_sentiment([
    "What a wonderful movie",
    "This was a disaster"
])

for item in sample_results:
    print("\nText:", item["text"])

    print("HF Label:", item["hf_label"])

    print("Numeric Label:", item["numeric_label"])

    print("Score:", item["score"])

## Your task:

As we can see for the huggingface sentiment analysis task,

with:

In [ ]:
classifier = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    tokenizer="distilbert-base-uncased-finetuned-sst-2-english",
    framework="pt"
)

the model is not strong enough (88% of accuracy for the first 50 reviews, just in per with Bag Of Word + Logistic Regression). the model is kind of small and out-dated.

So, i would like you guys to check out other models.

SEND YOUR FINDINGS HERE IN THE WECHAT GROUP, the model and the accuracy.

Ideally, we should be able to find a good model with, let's say, higher than 95% of accuracy.

#### Feedback from [one student](https://github.com/ceilf6/machine-learning/blob/main/tasks/session-400/results.txt):

Rank | Accuracy | Model Name  
-----|----------|----------------------------------  
 1   | 100.00%  | textattack/roberta-base-imdb          (>95%)  
 2   |  94.00%  | siebert/sentiment-roberta-large-english  
 3   |  94.00%  | lvwerra/distilbert-imdb  
 4   |  92.00%  | textattack/roberta-base-SST-2  
 5   |  92.00%  | aychang/roberta-base-imdb  
 6   |  90.00%  | textattack/distilbert-base-uncased-imdb  
 7   |  88.00%  | distilbert-base-uncased-finetuned-sst-2-english  (baseline)  
 8   |  86.00%  | textattack/bert-base-uncased-SST-2  
 9   |  86.00%  | textattack/albert-base-v2-SST-2  
10   |  82.00%  | textattack/distilbert-base-cased-SST-2  
11   |  80.00%  | nlptown/bert-base-multilingual-uncased-sentiment  
12   |  74.00%  | philschmid/distilbert-base-multilingual-cased-sentiment-2  
13   |  72.00%  | finiteautomata/bertweet-base-sentiment-analysis  
14   |  70.00%  | cardiffnlp/twitter-roberta-base-sentiment-latest  
15   |  68.00%  | lxyuan/distilbert-base-multilingual-cased-sentiments-student  
16   |  60.00%  | cardiffnlp/twitter-roberta-base-sentiment  
17   |  52.00%  | ProsusAI/finbert